# Chunking Strategy Comparison

Compares three strategies on the same 4 PDFs + 15 test queries:
- **Fixed-256**: fixed-size sliding window, ~256 words per chunk
- **Fixed-512**: fixed-size sliding window, ~512 words per chunk  
- **Section-aware**: our production chunker (respects headers, keeps tables intact)

Metric: **retrieval hit rate** — does at least one of the top-5 retrieved chunks come from an expected page?

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
sys.stdout.reconfigure(encoding='utf-8', errors='replace')

import json
import re
import numpy as np
import faiss
from dataclasses import dataclass, field
from sentence_transformers import SentenceTransformer
from src.ingestion.pdf_parser import PDFParser
from src.ingestion.chunker import SectionAwareChunker, Chunk

PDF_DIR = '../data/sample_pdfs'
TEST_SET_PATH = '../src/evaluation/test_set.json'
EMBED_MODEL = 'all-MiniLM-L6-v2'
TOP_K = 5  # retrieve top-5 for hit-rate measurement

print('Parsing PDFs...')
parser = PDFParser(remove_headers_footers=True)
documents = parser.parse_directory(PDF_DIR)
print(f'  {len(documents)} documents, {sum(d.total_pages for d in documents)} total pages')

with open(TEST_SET_PATH) as f:
    test_set = json.load(f)
print(f'  {len(test_set)} test queries loaded')

In [ ]:
# ── Fixed-size chunker (word-count based, with overlap) ──────────────────────

def fixed_size_chunks(documents, chunk_words: int, overlap_words: int = 32) -> list[Chunk]:
    """Slide a fixed-size window over each page's text."""
    chunks = []
    for doc in documents:
        for page in doc.pages:
            words = page.text.split()
            if not words:
                continue
            start = 0
            chunk_idx = 0
            while start < len(words):
                end = min(start + chunk_words, len(words))
                text = ' '.join(words[start:end])
                if len(text) >= 80:  # skip trivially short chunks
                    chunks.append(Chunk(
                        text=text,
                        metadata={
                            'page_number': page.page_number,
                            'filename': doc.filename,
                            'section': f'page-{page.page_number}',
                            'content_type': 'text',
                        },
                        chunk_id=f'{doc.filename}__fixed{chunk_words}_{page.page_number}_{chunk_idx:03d}',
                    ))
                    chunk_idx += 1
                start += chunk_words - overlap_words
    return chunks

In [ ]:
# ── In-memory FAISS index ────────────────────────────────────────────────────

model = SentenceTransformer(EMBED_MODEL)

def build_index(chunks: list[Chunk]):
    """Embed chunks and return a FAISS index + document list."""
    texts = [c.text for c in chunks]
    print(f'    Embedding {len(texts)} chunks...', end=' ', flush=True)
    embs = model.encode(texts, batch_size=64, show_progress_bar=False).astype(np.float32)
    faiss.normalize_L2(embs)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    print('done')
    return index, chunks

def search(query: str, index, chunks: list[Chunk], top_k: int = TOP_K) -> list[int]:
    """Return list of retrieved page numbers."""
    q_emb = model.encode([query], show_progress_bar=False).astype(np.float32)
    faiss.normalize_L2(q_emb)
    _, idxs = index.search(q_emb, min(top_k, len(chunks)))
    return [chunks[i].metadata['page_number'] for i in idxs[0] if i != -1]

def hit_rate(index, chunks, test_set, top_k=TOP_K) -> tuple[float, list[bool]]:
    """Fraction of queries where at least one expected page is in top-k."""
    hits = []
    for t in test_set:
        pages = search(t['question'], index, chunks, top_k)
        hits.append(bool(set(pages) & set(t.get('expected_pages', []))))
    return sum(hits) / len(hits), hits

In [ ]:
# ── Run all three strategies ─────────────────────────────────────────────────

results = {}

# Strategy 1: Fixed-256
print('Strategy: Fixed-256')
chunks_256 = fixed_size_chunks(documents, chunk_words=256, overlap_words=32)
print(f'  {len(chunks_256)} chunks')
idx_256, _ = build_index(chunks_256)
rate_256, hits_256 = hit_rate(idx_256, chunks_256, test_set)
results['Fixed-256'] = {'rate': rate_256, 'hits': hits_256, 'n_chunks': len(chunks_256)}
print(f'  Hit rate: {rate_256:.1%}\n')

# Strategy 2: Fixed-512
print('Strategy: Fixed-512')
chunks_512 = fixed_size_chunks(documents, chunk_words=512, overlap_words=64)
print(f'  {len(chunks_512)} chunks')
idx_512, _ = build_index(chunks_512)
rate_512, hits_512 = hit_rate(idx_512, chunks_512, test_set)
results['Fixed-512'] = {'rate': rate_512, 'hits': hits_512, 'n_chunks': len(chunks_512)}
print(f'  Hit rate: {rate_512:.1%}\n')

# Strategy 3: Section-aware (production)
print('Strategy: Section-Aware (production)')
chunker = SectionAwareChunker(chunk_size=512, chunk_overlap=64, min_chunk_size=100)
chunks_sa = []
for doc in documents:
    chunks_sa.extend(chunker.chunk_document(doc.pages, doc.metadata))
print(f'  {len(chunks_sa)} chunks')
idx_sa, _ = build_index(chunks_sa)
rate_sa, hits_sa = hit_rate(idx_sa, chunks_sa, test_set)
results['Section-Aware'] = {'rate': rate_sa, 'hits': hits_sa, 'n_chunks': len(chunks_sa)}
print(f'  Hit rate: {rate_sa:.1%}\n')

print('\nSummary:')
for name, r in results.items():
    print(f'  {name:20s}: {r["rate"]:.1%}  ({r["n_chunks"]} chunks)')

In [ ]:
# ── Bar chart ────────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

strategies  = list(results.keys())
rates       = [results[s]['rate'] * 100 for s in strategies]
n_chunks    = [results[s]['n_chunks'] for s in strategies]
colors      = ['#6baed6', '#3182bd', '#08519c']

fig, ax1 = plt.subplots(figsize=(8, 5))

bars = ax1.bar(strategies, rates, color=colors, width=0.5, zorder=3)
ax1.set_ylabel('Retrieval Hit Rate (%)', fontsize=12)
ax1.set_ylim(0, 110)
ax1.yaxis.grid(True, linestyle='--', alpha=0.6, zorder=0)
ax1.set_axisbelow(True)

# Value labels on bars
for bar, rate, n in zip(bars, rates, n_chunks):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1.5,
        f'{rate:.1f}%\n({n} chunks)',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

ax1.set_title('Chunking Strategy vs Retrieval Hit Rate\n(top-5 retrieval, 15 test queries)', fontsize=13)
ax1.set_xlabel('Chunking Strategy', fontsize=12)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../data/chunking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to data/chunking_comparison.png')

In [ ]:
# ── Per-query breakdown ──────────────────────────────────────────────────────

print(f'{"Question":<55} {"F-256":>6} {"F-512":>6} {"Sect":>6}')
print('-' * 76)
for i, t in enumerate(test_set):
    q = t['question'][:53]
    h256 = 'HIT' if hits_256[i] else 'miss'
    h512 = 'HIT' if hits_512[i] else 'miss'
    hsa  = 'HIT' if hits_sa[i]  else 'miss'
    print(f'{q:<55} {h256:>6} {h512:>6} {hsa:>6}')